In [ ]:
%matplotlib inline

import os
import pandas as pd
import numpy as np
from scipy.io import mmread
import scipy.sparse as sp
import matplotlib.pyplot as plt
from IPython.display import Image
import scanpy as sc
from cnmf import cNMF
rc_parms = {"figure.figsize": [5, 5], "figure.dpi": 500, "font.size": 10, "font.family": "Arial"}
save_parms = {"bbox_inches": "tight", "transparent": True}
if not os.path.exists('Thymus-Z1'):
    os.mkdir('Thymus-Z1')
    
np.random.seed(14)

In [ ]:
adata_gex = sc.read('../hvae/outputs/Thymus_HierarVi.h5ad')
adata_gex

In [ ]:
adata = sc.AnnData(adata_gex.obsm['RNA_Z1_denoised'])
adata.obsm["X_umap"] = adata_gex.obsm["Zc_umap"]
adata.obs = adata_gex.obs[['annotations', 'Genotype', 'batch']]

In [ ]:
mapp_to_genes = dict(zip(adata.var_names.to_series(), adata_gex.var_names.to_series()))

In [ ]:
pd.DataFrame(mapp_to_genes.items(), columns=['index', 'genes']).to_csv("./Thymus-Z1/gene-mapping.csv", index = False, header=False)

In [ ]:
adata.write('Thymus-Z1/counts.h5ad')

In [ ]:
adata = sc.read_h5ad("./Thymus-Z1/counts.h5ad")

In [ ]:
numiter=100 # Number of NMF replicates. Set this to a larger value ~200 for real data. We set this to a relatively low value here for illustration at a faster speed
numhvgenes=5125 ## Number of over-dispersed genes to use for running the actual factorizations

## Results will be saved to [output_directory]/[run_name] which in this example is example_PBMC/cNMF/pbmc_cNMF
output_directory = 'Thymus-Z1/cNMF'
if not os.path.exists(output_directory):
    os.mkdir(output_directory)
run_name = 'bc_cNMF'

## Specify the Ks to use as a space separated list in this case "5 6 7 8 9 10"
K = ' '.join([str(i) for i in range(5,23)])

## To speed this up, you can run it for only K=7-8 with the option below
#K = ' '.join([str(i) for i in range(7,9)])


seed = 14 ## Specify a seed pseudorandom number generation for reproducibility

## Path to the filtered counts dataset we output previously
countfn = 'Thymus-Z1/counts.h5ad'

In [ ]:
cnmf_obj = cNMF(output_dir=output_directory, name=run_name)

In [ ]:
cnmf_obj.prepare(counts_fn=countfn, components=np.arange(5,23), n_iter=numiter, seed=seed, num_highvar_genes=numhvgenes)

In [ ]:
cnmf_obj.factorize_multi_process(30)
cnmf_obj.combine()

In [ ]:
cnmf_obj.k_selection_plot(close_fig=False)
print('This saves the corresponding figure to the following file: %s' % cnmf_obj.paths['k_selection_plot'])

In [ ]:
# selected_K = 19
# density_threshold = 0.05

In [ ]:
selected_K = 11
density_threshold = 0.02

In [ ]:
cnmf_obj.consensus(k=selected_K, density_threshold=density_threshold, show_clustering=True, close_clustergram_fig=False)

In [ ]:
! ls ./Thymus-Z1/cNMF/bc_cNMF

In [ ]:
adata = sc.read(countfn)
adata.obsm["X_umap"] = adata_gex.obsm["Zc_umap"]
adata.obs = adata_gex.obs[['annotations', 'Genotype', 'batch']]

In [ ]:
hvgs = open('./Thymus-Z1/cNMF/bc_cNMF/bc_cNMF.overdispersed_genes.txt').read().split('\n')
hvgs

In [ ]:
adata.raw = sc.pp.log1p(adata.copy(), copy=True)

In [ ]:
adata

In [ ]:
adata = adata[:,hvgs]

In [ ]:
# sc.pp.scale(adata)
# sc.pp.pca(adata)
# sc.pl.pca_variance_ratio(adata, log=True)
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)

In [ ]:
usage_norm, gep_scores, gep_tpm, topgenes = cnmf_obj.load_results(K=selected_K, density_threshold=density_threshold)
usage_norm.columns = ['Program-%d' % i for i in usage_norm.columns]
# usage_file = cnmf_obj.paths['consensus_usages__txt'] % (selected_K, '0_8')
# gene_scores_file = cnmf_obj.paths['gene_spectra_score__txt'] % (selected_K, '0_8')
# gene_tpm_file = cnmf_obj.paths['gene_spectra_tpm__txt'] % (selected_K, '0_8')

In [ ]:
usage_norm

In [ ]:
topgenes = topgenes.replace(mapp_to_genes)

In [ ]:
topgenes.to_csv("./Thymus-Z1/cNMF/top_genes.csv")

In [ ]:
adata.obs = pd.merge(left=adata.obs, right=usage_norm, how='left', left_index=True, right_index=True)


In [ ]:
to_plot = usage_norm.columns.to_list()
to_plot.extend(['annotations'])

In [ ]:
#0.05
sc.pl.umap(adata, color=to_plot, ncols=3, vmin=0, vmax=1)

In [ ]:
with plt.rc_context(rc_parms):
    ax = sc.pl.umap(adata, color = "annotations", frameon = False, return_fig = True, title = '')#, legend_loc = None)
    plt.savefig("./Thymus-Z1/figures/Data_Umap_annotaions.png", **save_parms)


In [ ]:
sc.pl.dotplot(adata, {'genes':usage_norm.columns}, "annotations", dendrogram=True)

In [ ]:
import anndata
import matplotlib.ticker as ticker

adata_usage = anndata.AnnData(adata.obs[usage_norm.columns], obsm = {'X_umap':adata.obsm['X_umap']}, 
                              obs= adata.obs[['annotations']]
                             )
adata_usage = adata_usage[~adata_usage.obs["annotations"].isin(["Dying","Doublet"]),:]
adata_usage.obs['annotations'] = adata_usage.obs['annotations'].astype(str).astype("category")
adata_usage.obs['annotations'] = adata_usage.obs['annotations'].cat.reorder_categories(
    ['DP (Q1)', 'DP (Q2)', 'DP (Sig.)', "Neg. sel. (1)",'Neg. sel. (2)', 'Immature CD4', 'Immature CD8', 
    'Interferon sig.', 'Mature CD4', 'Mature CD8', 'Mature cycling', 'Treg', 'NKT', 'DN', 'DP (P)', 'GD T', 'B', 'Myeloid', "Erythrocyte"])
with plt.rc_context(rc_parms):
    sc.pl.heatmap(adata_usage, var_names = usage_norm.columns, groupby='annotations', show=False)
    # Get current axis
    ax = plt.gca()
    ax.set_ylabel("")
    plt.savefig("./Thymus-Z1/figures/Usages_heatmap-11.png", **save_parms)
    # sc.pl.heatmap(adata_usage, var_names = usage_norm.columns, groupby='annotations', dendrogram=True, show = False)
    # plt.savefig("./Thymus-Z1/figures/Usages_heatmap_dg-11.png", **save_parms)

In [ ]:
import anndata
adata_usage = anndata.AnnData(adata.obs[usage_norm.columns], obsm = {'X_umap':adata.obsm['X_umap']}, 
                              obs= adata.obs[['annotations']]
                             )
positive_sel = sc.read_h5ad("../Trajectory-Pos/Margaret/pos-margaret-0.3.h5ad")
adata_usage = adata_usage[positive_sel.obs_names,:]
with plt.rc_context(rc_parms):
    sc.pl.heatmap(adata_usage, var_names = usage_norm.columns, groupby='annotations', dendrogram=False, show = False)
    plt.savefig("./Thymus-Z1/figures/Usages_heatmap_dg_pos.pdf", **save_parms)